# Plant Disease Classifier: Training and Evaluation

Train an ImageNet-initialized ResNet18 on 21 leaf image classes covering corn, peach, bell pepper, potato, and tomato. The best validation checkpoint is exported as a PyTorch state dictionary for the Django application.

This notebook targets **Google Colab**. Place `PLANTS.zip` in `MyDrive/Colab_Projects/` and run the cells in order. The archive must contain `PLANTS/train/<class>/` and `PLANTS/valid/<class>/`. Each split must use the same class folders. Update the paths below if your Drive layout differs.

Training data is not included in this repository. Validation scores measure performance on this split and do not establish field accuracy. Saved cell outputs are cleared so each run records its own results.


## Environment and Dataset

Mount Google Drive, extract the dataset into the Colab runtime, and install the training dependencies.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects
!unzip -q -o PLANTS.zip -d /content/

from pathlib import Path
for split in ('train', 'valid'):
    split_path = Path('/content/PLANTS') / split
    print(f'{split}: {len(list(split_path.iterdir()))} class folders')


In [ ]:
%pip install torch torchvision matplotlib scikit-learn seaborn Pillow


## Libraries


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os
import time
import copy
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


## Configuration

The run uses a batch size of 64, 20 epochs, and Adam with a learning rate of 0.001. Freeze the feature extractor to train only the final layer. Reduce the batch size if GPU memory is limited.


In [ ]:
TRAIN_DATA_DIR = '/content/PLANTS/train'
VAL_DATA_DIR = '/content/PLANTS/valid'

MODEL_SAVE_PATH = '/content/drive/MyDrive/Colab_Projects/plant_classifier_model.pth'
BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
FREEZE_FEATURE_EXTRACTOR = True
IMAGE_SIZE = 224

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Data Loading and Preprocessing

Training uses random crops, horizontal flips, and color jitter. Validation uses resize and center crop, followed by ImageNet normalization; the web predictor must use the same validation transforms.


In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_dataset = datasets.ImageFolder(root=TRAIN_DATA_DIR, transform=data_transforms['train'])
val_dataset = datasets.ImageFolder(root=VAL_DATA_DIR, transform=data_transforms['val'])

class_names = train_dataset.classes
num_classes = len(class_names)
assert train_dataset.class_to_idx == val_dataset.class_to_idx, 'Class mappings differ between splits'
assert num_classes == 21, 'The web checkpoint expects 21 classes'
print(f"Found {num_classes} classes: {class_names}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=os.cpu_count())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count())

dataloaders = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}


## Model Architecture

Replace the pretrained ResNet18 classifier with one output per dataset class.


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze before replacing fc so the new classifier remains trainable.
if FREEZE_FEATURE_EXTRACTOR:
    for param in model.parameters():
        param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)


## Loss and Optimizer

Optimize cross-entropy loss. The parameter selection follows `FREEZE_FEATURE_EXTRACTOR`.


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters() if FREEZE_FEATURE_EXTRACTOR else model.parameters(), lr=LEARNING_RATE)


## Training Loop

Track loss and accuracy for each split and restore the checkpoint with the highest validation accuracy.


In [ ]:
def train_model(model, criterion, optimizer, dataloaders, dataset_sizes, num_epochs=NUM_EPOCHS):
    since = time.time()

    # Deep-copy tensors so later optimizer steps cannot overwrite the best checkpoint.
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'train':
                train_losses.append(epoch_loss)
                train_accuracies.append(epoch_acc.item())
            else:
                val_losses.append(epoch_loss)
                val_accuracies.append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model, train_losses, val_losses, train_accuracies, val_accuracies


## Run Training

Run both phases for the configured number of epochs.


In [ ]:
print("Starting training...")
model_ft, train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model, criterion, optimizer, dataloaders, dataset_sizes, num_epochs=NUM_EPOCHS
)


## Export Checkpoint

Save the selected state dictionary to Google Drive. Copy this file to `models/plant_classifier_model.pth` in the Django project after training.


In [ ]:
torch.save(model_ft.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")


## Training History

Plot loss and accuracy for training and validation.


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.title('Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()


## Validation Evaluation

Evaluate the selected checkpoint on the validation split. This is the same split used for checkpoint selection; it is not an independent test set.


In [ ]:
print("\n--- Final Evaluation on Validation Set ---")
model_ft.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model_ft(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f'Accuracy on the validation set: {100 * correct / total:.2f}%')

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

conf_mat = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(conf_mat)

plt.figure(figsize=(10, 8))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


## Single-Image Inference

Reload the exported checkpoint and upload a leaf image through Colab. The output class order must match the training folders and `classifier/predictor.py`. Confidence is a softmax score over the supported labels, not a calibrated probability of a correct diagnosis.


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

MODEL_PATH = '/content/drive/MyDrive/Colab_Projects/plant_classifier_model.pth'
IMAGE_SIZE = 224

CLASS_NAMES = [
    'Corn (maize)__Cercospora_leaf_spot Gray_leaf_spot',
    'Corn (maize)__Common_rust',
    'Corn (maize)__Northern_Leaf_Blight',
    'Corn (maize)__healthy',
    'Peach__Bacterial_spot',
    'Peach__healthy',
    'Pepper,bell__Bacterial_spot',
    'Pepper,bell__healthy',
    'Potato__Early_blight',
    'Potato__Late_blight',
    'Potato__healthy',
    'Tomato__Bacterial_spot',
    'Tomato__Early_blight',
    'Tomato__Late_blight',
    'Tomato__Leaf_Mold',
    'Tomato__Septoria_leaf_spot',
    'Tomato__Spider_mites Two-spotted_spider_mite',
    'Tomato__Target_Spot',
    'Tomato__Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato__Tomato_mosaic_virus',
    'Tomato__healthy'
]
# ImageFolder assigns output indices from sorted folder names.
CLASS_NAMES.sort()
assert CLASS_NAMES == [name.replace('___', '__') for name in class_names], (
    'Update CLASS_NAMES and classifier/predictor.py to match the training class order'
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.resnet18(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(CLASS_NAMES))

try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    model.eval()
    model.to(device)
    print(f"Model loaded successfully from {MODEL_PATH}")
except FileNotFoundError:
    print(f"Error: Model file not found at {MODEL_PATH}. Did you train and save it?")
    model = None
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict_image(image_path):
    if model is None:
        print("Model not loaded, cannot predict.")
        return None, None
    try:
        image = Image.open(image_path).convert('RGB')
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
        return None, None
    except Exception as e:
        print(f"Error opening image {image_path}: {e}")
        return None, None

    input_tensor = preprocess(image)
    input_batch = input_tensor.unsqueeze(0)

    input_batch = input_batch.to(device)

    with torch.no_grad():
        output = model(input_batch)

    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top_prob, top_idx = torch.topk(probabilities, 1)

    predicted_label = CLASS_NAMES[top_idx.item()]
    confidence = top_prob.item() * 100

    return predicted_label, confidence

if __name__ == '__main__':
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Upload an image before running prediction.')
    test_image_filename = next(iter(uploaded))

    print(f"\nTesting with uploaded image: {test_image_filename}")
    predicted_label, confidence = predict_image(test_image_filename)

    if predicted_label:
        print(f"Predicted Label: {predicted_label}")
        print(f"Confidence: {confidence:.2f}%")
